## Train with MLflow Autolog

### Purpose

This notebook demonstrates how to instrument an existing Databricks training workflow with MLflow.

It is the recommended entry point for teams that:
- Already train models in Databricks
- Use a standard framework (scikit-learn, XGBoost, etc.)
- Are not yet using MLflow tracking or model registration

After running this notebook, the model is registered in Unity Catalog and ready for the deployment pipeline (evaluation → approval → serving).

### What changes from your current training notebook

| Before | After |
|---|---|
| Train and export pickle manually | `mlflow.autolog()` captures everything automatically |
| No experiment tracking | Params, metrics, and model logged to MLflow Experiment |
| No model registry | Model registered to Unity Catalog with version and alias |
| Serve outside Databricks | Ready for Databricks Model Serving (Notebook 4a) |

### What this notebook does

1. Loads training data from Unity Catalog
2. Trains a scikit-learn model with `mlflow.autolog()` enabled
3. Registers the logged model to Unity Catalog Model Registry
4. Sets task values for downstream notebooks (evaluation → approval → serving)

Approval and Champion alias are set in `3_model_approval.ipynb` after evaluation.

In [0]:
%pip install -q threadpoolctl>=3.5.0
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [0]:
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("features_table_name", "iris_features", "Features Table Name")
dbutils.widgets.text("labels_table_name", "iris_labels", "Labels Table Name")
dbutils.widgets.text("model_name", "native_sklearn", "Model Name")
dbutils.widgets.text("experiment_name", "/Shared/byo_model_experiment", "MLflow Experiment Name")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
features_table_name = dbutils.widgets.get("features_table_name")
labels_table_name = dbutils.widgets.get("labels_table_name")
model_name = dbutils.widgets.get("model_name")
experiment_name = dbutils.widgets.get("experiment_name")

registered_model_name = f"{catalog}.{schema}.{model_name}"

### Load training data from Unity Catalog

Reads feature and label tables from Unity Catalog.

Replace `features_table_name` and `labels_table_name` with your own tables.

If running end-to-end from `0_export_model_and_artifacts`, these tables are already created.

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split

features_sdf = spark.read.table(f"{catalog}.{schema}.{features_table_name}")
labels_sdf = spark.read.table(f"{catalog}.{schema}.{labels_table_name}")

X = features_sdf.drop("id").toPandas()
y = labels_sdf.drop("id").toPandas().values.ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Features: {list(X.columns)}")

### Train with MLflow autolog

`mlflow.autolog()` automatically captures:
- **Params** — all hyperparameters passed to the estimator
- **Metrics** — training score
- **Model artifact** — serialized model logged to the run
- **Feature importance** — for tree-based models
- **Signature** — inferred from the input example

This replaces manual `mlflow.log_param()`, `mlflow.log_metric()`, and `mlflow.sklearn.log_model()` calls.

Your training code does not change.

In [0]:
import mlflow
from sklearn.ensemble import RandomForestClassifier

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_name)

# Enable autolog — one line. Everything below is unchanged from your existing training code.
mlflow.autolog(log_input_examples=True, log_model_signatures=True)

with mlflow.start_run(run_name=f"train_{model_name}") as run:
    # --- Your existing training code (unchanged) ---
    clf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
    clf.fit(X_train, y_train)
    # --- End of your training code ---

    run_id = run.info.run_id

print(f"Run ID: {run_id}")
print(f"Experiment: {experiment_name}")
print("Open the Experiment UI (left sidebar → Experiments) to see logged params, metrics, and model artifact.")

### Register to Unity Catalog Model Registry

Registers the logged model artifact to Unity Catalog.

The model version is created in `Pending` state. Approval and Champion alias are assigned
by `3_model_approval.ipynb` after evaluation passes thresholds.

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# autolog logs the model as a directory artifact alongside files like estimator.html —
# use is_dir to target the model directory specifically
artifacts = client.list_artifacts(run_id)
model_artifact_path = next(
    (a.path for a in artifacts if a.is_dir),
    "model"
)
model_uri = f"runs:/{run_id}/{model_artifact_path}"

result = mlflow.register_model(model_uri=model_uri, name=registered_model_name)
version = str(result.version)

print(f"Registered: {registered_model_name} v{version}")
print("Next: run 2_model_evaluation.ipynb to evaluate, then 3_model_approval.ipynb to promote to Champion.")

In [0]:
# Pass values to downstream job tasks (evaluation → approval → serving)
dbutils.jobs.taskValues.set(key="model_version", value=version)
dbutils.jobs.taskValues.set(key="model_name", value=registered_model_name)